# Historical financial analysis: learn by running the code

Run the cells from top to bottom. The first example is synthetic; the Tega example later contains only two historical income-statement periods. The code uses the standard library and the project package.


## Find the project and import the functions

Open this notebook from the repository or its `notebooks` folder. The cell locates `src` and makes the package importable without installation.


In [1]:
import sys
from pathlib import Path

project = next(
    (
        p
        for p in (Path.cwd(), Path.cwd().parent)
        if (p / "src/equity_analytics/financials").is_dir()
    ),
    None,
)
if project is None:
    raise RuntimeError("Open this notebook from the project or its notebooks folder.")
sys.path.insert(0, str(project / "src"))

from equity_analytics.financials import analyze_history, load_history
from equity_analytics.financials.__main__ import format_metric

print("Financial analysis functions loaded.")

Financial analysis functions loaded.


## Load and inspect five annual records

`load_history` reads the JSON and validates each data class. The result keeps its units and source notes. Five annual observations provide four annual growth intervals.


In [2]:
history = load_history(project / "examples/demo_financial_history.json")
print(history.company.name, "|", history.company.data_kind)
for annual in history.annuals:
    print(
        f"FY{annual.fiscal_year}: revenue {annual.revenue:,.2f} {history.company.currency} {history.company.financial_unit}"
    )

Demo Industrials Ltd | synthetic
FY2022: revenue 6,600.00 INR crore
FY2023: revenue 7,300.00 INR crore
FY2024: revenue 8,100.00 INR crore
FY2025: revenue 9,000.00 INR crore
FY2026: revenue 10,000.00 INR crore


## Follow the inputs into the ratios

Each result is a `Metric`: value, unit, and a reason if it is unavailable. Percentages are stored as fractions.


In [3]:
report = analyze_history(history)
latest = history.annuals[-1]
metrics = report.years[-1].metrics
print(f"EBIT: {latest.ebit:,.2f}; revenue: {latest.revenue:,.2f}")
for name in (
    "revenue_growth",
    "ebit_margin",
    "roe",
    "net_debt_to_ebitda",
    "cash_flow_after_capex",
):
    print(f"{name}: {format_metric(metrics[name])}")
print(
    f"Revenue CAGR: {format_metric(report.revenue_cagr)} over {report.cagr_intervals} years"
)

EBIT: 1,500.00; revenue: 10,000.00
revenue_growth: 11.11%
ebit_margin: 15.00%
roe: 20.30%
net_debt_to_ebitda: 0.41x
cash_flow_after_capex: 830.00
Revenue CAGR: 10.95% over 4 years


## Explain an unavailable return

ROE needs opening and closing owners' equity. The first annual observation lacks an opening balance sheet, so the engine returns a reason instead of making up a denominator.


In [4]:
first_roe = report.years[0].metrics["roe"]
print("First-year ROE:", first_roe.value)
print("Reason:", first_roe.reason)

First-year ROE: None
Reason: requires immediately preceding year-end balance


## Change one input and predict the effects

Create a modified copy in memory. Changing EBIT affects operating ratios; historical revenue and net income remain separate supplied inputs. This cell does not edit the JSON file.


In [5]:
from dataclasses import replace

changed_annual = replace(latest, ebit=1600.0)
changed_history = replace(history, annuals=history.annuals[:-1] + (changed_annual,))
changed = analyze_history(changed_history).years[-1].metrics
for name in ("ebit_margin", "ebitda", "roce", "finance_cost_coverage", "net_margin"):
    print(f"{name}: {format_metric(metrics[name])} -> {format_metric(changed[name])}")

ebit_margin: 15.00% -> 16.00%
ebitda: 1,850.00 -> 1,950.00
roce: 24.29% -> 25.91%
finance_cost_coverage: 11.72x -> 12.50x
net_margin: 10.00% -> 10.00%


## Try the sourced Tega example

These are FY2024 and FY2025 income statements in INR million, based on the source presentations identified in the JSON. Read the normalization notes before comparing EBITDA. This example has no balance-sheet or cash-flow data.


In [6]:
tega = load_history(project / "examples/tega_fy2024_fy2025_income_example.json")
tega_report = analyze_history(tega)
print(
    tega.company.name, "| amounts:", tega.company.currency, tega.company.financial_unit
)
for year in tega_report.years:
    print(
        f"FY{year.fiscal_year}: operating EBITDA margin {format_metric(year.metrics['ebitda_margin'])}"
    )
print("ROE:", tega_report.years[-1].metrics["roe"].reason)
print("CFO less capex:", tega_report.years[-1].metrics["cash_flow_after_capex"].reason)

Tega Industries Limited | amounts: INR million
FY2024: operating EBITDA margin 21.17%
FY2025: operating EBITDA margin 20.74%
ROE: missing opening or closing balance
CFO less capex: missing input


## Where to read next

- `src/equity_analytics/financials/models.py`: data classes and validation.
- `src/equity_analytics/financials/io.py`: JSON-to-object conversion.
- `src/equity_analytics/financials/ratios.py`: calculation loop and ratio rules.
- `docs/FINANCIAL_ANALYSIS_METHODS.md`: field meanings and formulas.

Exercise: predict what happens to the current ratio when current liabilities double, then create a modified synthetic annual record with `dataclasses.replace` and verify your answer.
